<div style="border:1px solid #d0d0d0; padding:25px; border-radius:8px; background:#f8f9fa">

<h1 style="margin-bottom:10px">
Estimación de temperatura superficial (LST) con Landsat 9  
Emisividad, parámetros atmosféricos, RTE e inversión de Planck
</h1>

<p>Material docente para la asignatura <b>Teledetección</b>.</p>

<b>Autor</b><br>
Julián Garzón Barrero<br>
Docente<br>
Programa de Ingeniería Topográfica y Geomática<br>
Universidad del Quindío<br>
juliangarzonb@uniquindio.edu.co

<hr>

<b>Apoyo técnico</b><br>
Estructuración y revisión computacional asistida mediante herramientas de inteligencia artificial (ChatGPT, OpenAI).

<hr>

<b>Propósito</b>

<p>
Reconstruir explícitamente un flujo físico de estimación de temperatura superficial terrestre (LST) a partir de radiancia térmica Landsat 9 Collection 2 Level-1, reflectancia óptica corregida mediante DOS1 y parámetros atmosféricos auxiliares disponibles en Collection 2 Level-2.
</p>

<p>
El cuaderno enfatiza la trazabilidad entre magnitudes físicas, ecuaciones y operaciones computacionales.
</p>

</div>

---

## 📘 Módulo 1 -- Carga y verificación de insumos

**Objetivo**  
Preparar el entorno de trabajo y comprobar que los archivos requeridos están disponibles y son espacialmente compatibles.

**Entradas**
- producto Landsat 9 Collection 2 Level-1;
- producto Landsat 9 Collection 2 Level-2;
- reflectancia óptica corregida mediante DOS1;
- archivo `MTL`.

**Proceso**
- monta Google Drive;
- define `BASE_PATH`;
- registra y verifica las rutas;
- inspecciona dimensiones, georreferenciación y sistema de referencia;
- carga el archivo de metadatos.

**Salida**  
Insumos y metadatos listos para el procesamiento.

In [ ]:
# ============================================================
# MÓDULO 1 — Carga de las escenas Landsat
# Objetivo:
# Cargar los datos Landsat necesarios para el flujo de estimación
# de temperatura superficial y verificar su disponibilidad y
# compatibilidad espacial.
# ============================================================

# -----------------------------
# 0. Montaje de Google Drive
# -----------------------------
from google.colab import drive
drive.mount("/content/drive")

# -----------------------------
# 1. Importación de librerías
# -----------------------------
import os
from pathlib import Path
import numpy as np
import rasterio
import matplotlib.pyplot as plt

# -----------------------------
# 2. Definición de rutas base
# -----------------------------
BASE_PATH = "/content/drive/MyDrive/Colab Notebooks/datos_LST"  # <-- modificar si es necesario
l1_folder = f"{BASE_PATH}/LC09_L1SP_009052_20250114_20250115_02_T1"
l2_folder = f"{BASE_PATH}/LC09_L2SP_009052_20250114_20250115_02_T1"

# -----------------------------
# 3. Rutas de archivos de entrada
# -----------------------------
l1_dos1_path = f"{l1_folder}/L9_DOS1_reflectance.tif"
b10_l1_path  = f"{l1_folder}/LC09_L1TP_009052_20250114_20250114_02_T1_B10.tif"

qa_pixel_path = f"{l2_folder}/LC09_L2SP_009052_20250114_20250115_02_T1_QA_PIXEL.TIF"
st_urad_path  = f"{l2_folder}/LC09_L2SP_009052_20250114_20250115_02_T1_ST_URAD.TIF"
st_drad_path  = f"{l2_folder}/LC09_L2SP_009052_20250114_20250115_02_T1_ST_DRAD.TIF"
st_atran_path = f"{l2_folder}/LC09_L2SP_009052_20250114_20250115_02_T1_ST_ATRAN.TIF"
st_b10_path   = f"{l2_folder}/LC09_L2SP_009052_20250114_20250115_02_T1_ST_B10.TIF"

mtl_path = f"{l2_folder}/LC09_L2SP_009052_20250114_20250115_02_T1_MTL.txt"

# -----------------------------
# 4. Diccionario central de entradas
# -----------------------------
input_files = {
    "L1_DOS1":  l1_dos1_path,
    "L1_B10":   b10_l1_path,
    "QA_PIXEL": qa_pixel_path,
    "ST_URAD":  st_urad_path,
    "ST_DRAD":  st_drad_path,
    "ST_ATRAN": st_atran_path,
    "ST_B10":   st_b10_path,
    "MTL":      mtl_path,
}

# -----------------------------
# 5. Verificación de existencia de archivos
# -----------------------------
print("=" * 70)
print("VERIFICACIÓN DE EXISTENCIA DE ARCHIVOS")
print("=" * 70)

missing_files = []

for key, path in input_files.items():
    exists = os.path.exists(path)
    print(f"{key:<10} -> {'OK' if exists else 'NO ENCONTRADO'}")
    if not exists:
        missing_files.append((key, path))

if missing_files:
    print("\nSe encontraron archivos faltantes:")
    for key, path in missing_files:
        print(f" - {key}: {path}")
    raise FileNotFoundError("Faltan uno o más archivos requeridos. Corrige las rutas antes de continuar.")

# -----------------------------
# 6. Función para extraer metadatos espaciales
# -----------------------------
def get_raster_info(path):
    with rasterio.open(path) as src:
        return {
            "path": path,
            "crs": src.crs,
            "shape": (src.height, src.width),
            "count": src.count,
            "dtype": src.dtypes,
            "transform": src.transform,
            "bounds": src.bounds,
            "res": src.res,
        }

# -----------------------------
# 7. Lectura de metadatos espaciales
# -----------------------------
raster_paths = {name: path for name, path in input_files.items() if name != "MTL"}
raster_info = {name: get_raster_info(path) for name, path in raster_paths.items()}

print("\n" + "=" * 70)
print("RESUMEN DE METADATOS ESPACIALES")
print("=" * 70)

for name, info in raster_info.items():
    print(f"\n{name}")
    print(f"  Ruta      : {info['path']}")
    print(f"  CRS       : {info['crs']}")
    print(f"  Shape     : {info['shape']}")
    print(f"  Bandas    : {info['count']}")
    print(f"  Dtype     : {info['dtype']}")
    print(f"  Bounds    : {info['bounds']}")
    print(f"  Resolución: {info['res']}")
    print(f"  Transform : {info['transform']}")

# -----------------------------
# 8. Verificación de compatibilidad espacial
#    Se toma QA_PIXEL como referencia geométrica del Level-2
# -----------------------------
print("\n" + "=" * 70)
print("VERIFICACIÓN DE COMPATIBILIDAD ESPACIAL")
print("=" * 70)

reference_name = "QA_PIXEL"
reference_info = raster_info[reference_name]

compatibility_report = {}

for name, info in raster_info.items():
    same_crs = info["crs"] == reference_info["crs"]
    same_shape = info["shape"] == reference_info["shape"]
    same_transform = info["transform"] == reference_info["transform"]

    compatibility_report[name] = {
        "same_crs": same_crs,
        "same_shape": same_shape,
        "same_transform": same_transform
    }

    print(f"\n{name}")
    print(f"  CRS igual a {reference_name:<8}: {same_crs}")
    print(f"  Shape igual a {reference_name:<8}: {same_shape}")
    print(f"  Transform igual a {reference_name}: {same_transform}")

# -----------------------------
# 9. Diagnóstico general de alineación
# -----------------------------
all_crs_ok = all(v["same_crs"] for v in compatibility_report.values())
all_shape_ok = all(v["same_shape"] for v in compatibility_report.values())
all_transform_ok = all(v["same_transform"] for v in compatibility_report.values())

print("\n" + "=" * 70)
print("DIAGNÓSTICO GENERAL")
print("=" * 70)
print(f"Todos comparten el mismo CRS       : {all_crs_ok}")
print(f"Todos comparten el mismo tamaño    : {all_shape_ok}")
print(f"Todos comparten la misma transform : {all_transform_ok}")

if all_crs_ok and all_shape_ok and all_transform_ok:
    print("\nResultado: compatibilidad espacial completa.")
else:
    print("\nResultado: se detectaron incompatibilidades espaciales.")
    print("Revise los archivos antes de continuar con operaciones píxel a píxel.")

# -----------------------------
# 10. Lectura del archivo MTL como texto
# -----------------------------
print("\n" + "=" * 70)
print("LECTURA DEL ARCHIVO MTL")
print("=" * 70)

with open(mtl_path, "r", encoding="utf-8", errors="ignore") as f:
    mtl_text = f.read()

print("MTL cargado correctamente.")
print(f"Número de caracteres leídos: {len(mtl_text)}")

print("\nMÓDULO 1 completado correctamente.")


---

## 📘 Módulo 2 -- Recuperación de metadatos radiométricos

**Objetivo**  
Extraer del archivo `MTL` los coeficientes necesarios para convertir B10 a radiancia y aplicar posteriormente la inversión de Planck.

**Parámetros**
- `RADIANCE_MULT_BAND_10`;
- `RADIANCE_ADD_BAND_10`;
- `K1_CONSTANT_BAND_10`;
- `K2_CONSTANT_BAND_10`.

**Salida**  
Coeficientes de calibración radiométrica y constantes térmicas de B10.

In [ ]:
# ============================================================
# MÓDULO 2 — Recuperación de metadatos radiométricos
# Objetivo:
# Extraer del metadato de la escena los parámetros necesarios
# para la calibración radiométrica térmica.
# ============================================================

# -----------------------------
# 1. Importación de librerías
# -----------------------------
import re

# -----------------------------
# 2. Función para extraer valores del MTL
# -----------------------------
def extract_mtl_value(mtl_text, key, cast_type=float):
    """
    Extrae el valor asociado a una clave del archivo MTL cargado como texto.

    Parámetros
    ----------
    mtl_text : str
        Contenido completo del archivo MTL.
    key : str
        Nombre exacto de la clave a buscar.
    cast_type : type
        Tipo al que se convertirá el valor extraído.

    Retorna
    -------
    cast_type
        Valor convertido al tipo solicitado.
    """
    pattern = rf"{key}\s=\s(.+)"
    match = re.search(pattern, mtl_text)

    if match is None:
        raise ValueError(f"No se encontró la clave '{key}' en el archivo MTL.")

    raw_value = match.group(1).strip().strip('"')

    try:
        return cast_type(raw_value)
    except Exception as e:
        raise ValueError(f"No fue posible convertir el valor de '{key}' ({raw_value}) a {cast_type}.") from e

# -----------------------------
# 3. Extracción de parámetros radiométricos
# -----------------------------
RADIANCE_MULT_BAND_10 = extract_mtl_value(mtl_text, "RADIANCE_MULT_BAND_10", float)
RADIANCE_ADD_BAND_10  = extract_mtl_value(mtl_text, "RADIANCE_ADD_BAND_10", float)

K1_CONSTANT_BAND_10   = extract_mtl_value(mtl_text, "K1_CONSTANT_BAND_10", float)
K2_CONSTANT_BAND_10   = extract_mtl_value(mtl_text, "K2_CONSTANT_BAND_10", float)

# -----------------------------
# 4. Resumen de resultados
# -----------------------------
print("=" * 70)
print("MÓDULO 2 — PARÁMETROS RADIOMÉTRICOS RECUPERADOS DEL MTL")
print("=" * 70)

print(f"RADIANCE_MULT_BAND_10 = {RADIANCE_MULT_BAND_10}")
print(f"RADIANCE_ADD_BAND_10  = {RADIANCE_ADD_BAND_10}")
print(f"K1_CONSTANT_BAND_10   = {K1_CONSTANT_BAND_10}")
print(f"K2_CONSTANT_BAND_10   = {K2_CONSTANT_BAND_10}")

# -----------------------------
# 5. Diccionario opcional de organización
# -----------------------------
radiometric_metadata = {
    "RADIANCE_MULT_BAND_10": RADIANCE_MULT_BAND_10,
    "RADIANCE_ADD_BAND_10":  RADIANCE_ADD_BAND_10,
    "K1_CONSTANT_BAND_10":   K1_CONSTANT_BAND_10,
    "K2_CONSTANT_BAND_10":   K2_CONSTANT_BAND_10,
}

print("\nParámetros almacenados en el diccionario: radiometric_metadata")

---

## 📘 Módulo 3 -- Conversión de DN a radiancia en el sensor

**Objetivo**  
Convertir los valores digitales de B10 a radiancia espectral registrada por el sensor.

$$
L_{\lambda}^{sen}=M_LQ_{cal}+A_L
$$

donde $L_{\lambda}^{sen}$ se expresa en $\mathrm{W\,m^{-2}\,sr^{-1}\,\mu m^{-1}}$.

**Proceso**
- lee B10;
- identifica `nodata`;
- construye una máscara térmica inicial;
- aplica los coeficientes radiométricos.

**Salida**  
Radiancia térmica observada por el sensor.

In [ ]:
# ============================================================
# MÓDULO 3 — Conversión de DN a radiancia en el sensor
# Objetivo:
# Convertir los valores digitales de la banda térmica B10 a
# radiancia espectral observada por el sensor.
# Resultado:
# Radiancia térmica en el sensor (L_lambda_sen)
# ============================================================

# -----------------------------
# 1. Importación de librerías
# -----------------------------

# -----------------------------
# 2. Lectura de la banda térmica B10 (Level-1)
# -----------------------------
with rasterio.open(b10_l1_path) as src:
    b10_dn = src.read(1).astype(np.float32)
    b10_profile = src.profile.copy()
    b10_transform = src.transform
    b10_crs = src.crs
    b10_nodata = src.nodata

# -----------------------------
# 3. Conversión de DN a radiancia en el sensor
#    L_lambda_sen = M_L * DN + A_L
# -----------------------------
b10_valid_mask = np.isfinite(b10_dn)
if b10_nodata is not None:
    b10_valid_mask &= (b10_dn != b10_nodata)

L_lambda_sen = np.full(b10_dn.shape, np.nan, dtype=np.float32)
L_lambda_sen[b10_valid_mask] = (
    RADIANCE_MULT_BAND_10 * b10_dn[b10_valid_mask] +
    RADIANCE_ADD_BAND_10
).astype(np.float32)

# -----------------------------
# 4. Estadísticas básicas de control
# -----------------------------
print("=" * 70)
print("MÓDULO 3 — CONVERSIÓN DE DN A RADIANCIA EN EL SENSOR")
print("=" * 70)

print(f"Shape de B10 DN            : {b10_dn.shape}")
print(f"Tipo de dato DN            : {b10_dn.dtype}")
print(f"Nodata B10                 : {b10_nodata}")
print(f"Píxeles válidos B10        : {np.sum(b10_valid_mask):,}")

print("\nEstadísticas de L_lambda_sen:")
print(f"  Mínimo                   : {np.nanmin(L_lambda_sen):.6f}")
print(f"  Máximo                   : {np.nanmax(L_lambda_sen):.6f}")
print(f"  Media                    : {np.nanmean(L_lambda_sen):.6f}")
print(f"  Desviación estándar      : {np.nanstd(L_lambda_sen):.6f}")

# -----------------------------
# 5. Perfil de salida para uso posterior
# -----------------------------
L_lambda_sen_profile = b10_profile.copy()
L_lambda_sen_profile.update(
    dtype="float32",
    count=1
)

print("\nVariables disponibles:")
print(" - b10_dn")
print(" - b10_valid_mask")
print(" - L_lambda_sen")
print(" - L_lambda_sen_profile")

## 📘 Módulo 4 -- Reflectancia óptica corregida (DOS1)

**Objetivo**  
Cargar las bandas ópticas B2–B7 previamente corregidas mediante DOS1 y comprobar su compatibilidad espacial.

**Proceso**
- identifica las bandas;
- lee las matrices de reflectancia;
- verifica dimensiones, transformación y sistema de referencia.

**Salida**  
Reflectancia óptica disponible para los cálculos espectrales.

In [ ]:
# ============================================================
# MÓDULO 4 — Reflectancia óptica corregida (DOS1)
# Objetivo:
# Cargar la imagen multibanda de reflectancia corregida mediante
# DOS1 y disponer sus bandas ópticas para uso posterior.
#
# Resultado:
# Stack de reflectancia corregida: refl_stack
# Bandas individuales: B2, B3, B4, B5, B6, B7
# ============================================================

# -----------------------------
# 1. Importación de librerías
# -----------------------------

# -----------------------------
# 2. Lectura de la imagen multibanda de reflectancia (DOS1)
#    Orden del stack esperado:
#    [B2, B3, B4, B5, B6, B7]
# -----------------------------
with rasterio.open(l1_dos1_path) as src:

    # Lectura del stack de reflectancia
    refl_stack = src.read().astype(np.float32)   # shape: (6, rows, cols)

    # Metadatos espaciales
    refl_profile   = src.profile.copy()
    refl_transform = src.transform
    refl_crs       = src.crs
    refl_nodata    = src.nodata
    refl_count     = src.count

# -----------------------------
# 3. Verificación básica del número de bandas
# -----------------------------
if refl_count != 6:
    raise ValueError(
        f"Se esperaban 6 bandas en refl_stack [B2, B3, B4, B5, B6, B7], "
        f"pero se encontraron {refl_count}."
    )

# -----------------------------
# 4. Asignación explícita de bandas
# -----------------------------
B2 = refl_stack[0, :, :]
B3 = refl_stack[1, :, :]
B4 = refl_stack[2, :, :]
B5 = refl_stack[3, :, :]
B6 = refl_stack[4, :, :]
B7 = refl_stack[5, :, :]

# -----------------------------
# 5. Resumen general del stack
# -----------------------------
print("=" * 70)
print("MÓDULO 4 — REFLECTANCIA ÓPTICA CORREGIDA (DOS1)")
print("=" * 70)

print(f"Ruta              : {l1_dos1_path}")
print(f"Bandas            : {refl_count}")
print(f"Shape             : {refl_stack.shape}")
print(f"CRS               : {refl_crs}")
print(f"Nodata            : {refl_nodata}")

print("\nRango global del stack:")
print(f"min = {np.nanmin(refl_stack):.6f}")
print(f"max = {np.nanmax(refl_stack):.6f}")

print("\nOrden de bandas esperado:")
print("[B2, B3, B4, B5, B6, B7]")



---

## 📘 Módulo 5 -- Definición del dominio terrestre con `QA_PIXEL`

**Objetivo**  
Excluir píxeles afectados por condiciones de calidad incompatibles con la estimación térmica.

**Proceso**
Se interpretan los bits relevantes de `QA_PIXEL` para construir una máscara común de píxeles válidos.

**Salida**  
Dominio espacial de análisis para las etapas posteriores.

In [ ]:
# ============================================================
# MÓDULO 5 — Definición del dominio terrestre a partir de QA_PIXEL
# Objetivo:
# Construir una máscara de validez observacional y una máscara
# terrestre excluyendo agua y píxeles afectados por fill, nubes,
# sombras y cirrus, utilizando únicamente la banda QA_PIXEL
# de Landsat Collection 2.
#
# Resultado:
# - qa_pixel
# - valid_obs_mask
# - water_mask
# - land_mask
# ============================================================

# -----------------------------
# 1. Importación de librerías
# -----------------------------

# -----------------------------
# 2. Verificación directa de compatibilidad espacial
#    entre reflectancia DOS1 y QA_PIXEL
# -----------------------------
with rasterio.open(l1_dos1_path) as src_l1, rasterio.open(qa_pixel_path) as src_qa:
    if src_l1.crs != src_qa.crs:
        raise ValueError("L1_DOS1 y QA_PIXEL no comparten el mismo CRS.")
    if src_l1.transform != src_qa.transform:
        raise ValueError("L1_DOS1 y QA_PIXEL no comparten la misma transform.")
    if (src_l1.height != src_qa.height) or (src_l1.width != src_qa.width):
        raise ValueError("L1_DOS1 y QA_PIXEL no comparten el mismo shape.")

# -----------------------------
# 3. Lectura de QA_PIXEL
# -----------------------------
with rasterio.open(qa_pixel_path) as src:
    qa_pixel = src.read(1)
    qa_profile = src.profile.copy()
    qa_transform = src.transform
    qa_crs = src.crs
    qa_nodata = src.nodata

# -----------------------------
# 4. Extracción de bits relevantes
#    Landsat Collection 2 QA_PIXEL
#    Bit 0 -> fill
#    Bit 1 -> dilated cloud
#    Bit 2 -> cirrus
#    Bit 3 -> cloud
#    Bit 4 -> cloud shadow
#    Bit 5 -> snow
#    Bit 7 -> water
# -----------------------------
fill_bit          = ((qa_pixel & (1 << 0)) != 0)
dilated_cloud_bit = ((qa_pixel & (1 << 1)) != 0)
cirrus_bit        = ((qa_pixel & (1 << 2)) != 0)
cloud_bit         = ((qa_pixel & (1 << 3)) != 0)
cloud_shadow_bit  = ((qa_pixel & (1 << 4)) != 0)
water_bit         = ((qa_pixel & (1 << 7)) != 0)

# -----------------------------
# 5. Máscara de validez observacional
#    Solo QA_PIXEL:
#    válido = no fill, no dilated cloud, no cirrus,
#             no cloud, no cloud shadow
# -----------------------------
valid_obs_mask = ~(
    fill_bit |
    dilated_cloud_bit |
    cirrus_bit |
    cloud_bit |
    cloud_shadow_bit
)

# -----------------------------
# 6. Máscara de agua
#    Solo QA_PIXEL:
#    agua = water_bit
# -----------------------------
water_mask = valid_obs_mask & water_bit

# -----------------------------
# 7. Dominio terrestre final
#    tierra = observación válida y no agua
# -----------------------------
land_mask = valid_obs_mask & (~water_bit)

# -----------------------------
# 8. Estadísticas de control
# -----------------------------
n_total   = qa_pixel.size
n_valid   = np.sum(valid_obs_mask)
n_water   = np.sum(water_mask)
n_land    = np.sum(land_mask)

print("=" * 72)
print("MÓDULO 5 — DOMINIO TERRESTRE USANDO SOLO QA_PIXEL")
print("=" * 72)

print(f"Shape QA_PIXEL                    : {qa_pixel.shape}")
print(f"Tipo de dato QA_PIXEL             : {qa_pixel.dtype}")
print(f"Nodata QA_PIXEL                   : {qa_nodata}")

print("\nConteo de bits activados:")
print(f"  Fill (bit 0)                    : {np.sum(fill_bit):,}")
print(f"  Dilated cloud (bit 1)           : {np.sum(dilated_cloud_bit):,}")
print(f"  Cirrus (bit 2)                  : {np.sum(cirrus_bit):,}")
print(f"  Cloud (bit 3)                   : {np.sum(cloud_bit):,}")
print(f"  Cloud shadow (bit 4)            : {np.sum(cloud_shadow_bit):,}")
print(f"  Water (bit 7)                   : {np.sum(water_bit):,}")

print("\nResumen de máscaras:")
print(f"  Total de píxeles                : {n_total:,}")
print(f"  valid_obs_mask = 1              : {n_valid:,} ({100*n_valid/n_total:.2f}%)")
print(f"  water_mask = 1                  : {n_water:,} ({100*n_water/n_total:.2f}%)")
print(f"  land_mask = 1                   : {n_land:,} ({100*n_land/n_total:.2f}%)")

# -----------------------------
# 9. Función de stretch para RGB
#    Solo para visualización
# -----------------------------
def stretch_band(band, mask=None, pmin=2, pmax=98):
    if mask is None:
        valid = np.isfinite(band)
    else:
        valid = np.isfinite(band) & mask

    if not np.any(valid):
        return np.zeros_like(band, dtype=np.float32)

    vmin, vmax = np.percentile(band[valid], (pmin, pmax))

    if vmax == vmin:
        return np.zeros_like(band, dtype=np.float32)

    out = (band - vmin) / (vmax - vmin)
    return np.clip(out, 0, 1)

# -----------------------------
# 10. Construcción de RGB
#     RGB = B4-B3-B2
#     Se usa valid_obs_mask solo para mejorar la visualización,
#     no para construir la máscara temática.
# -----------------------------
rgb = np.dstack((
    stretch_band(B4, mask=valid_obs_mask),
    stretch_band(B3, mask=valid_obs_mask),
    stretch_band(B2, mask=valid_obs_mask)
))

# -----------------------------
# 11. Overlay amarillo
#     píxeles excluidos del dominio terrestre
# -----------------------------
overlay_land = np.zeros((rgb.shape[0], rgb.shape[1], 4), dtype=np.float32)
overlay_land[..., 0] = 1.0
overlay_land[..., 1] = 1.0
overlay_land[..., 2] = 0.0
overlay_land[..., 3] = 0.0
overlay_land[~land_mask, 3] = 1.0

# -----------------------------
# 12. Visualización diagnóstica
# -----------------------------
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

axes[0, 0].imshow(rgb)
axes[0, 0].set_title("RGB color verdadero (B4-B3-B2)")
axes[0, 0].axis("off")

axes[0, 1].imshow(valid_obs_mask, cmap="gray")
axes[0, 1].set_title("valid_obs_mask (solo QA_PIXEL)")
axes[0, 1].axis("off")

axes[1, 0].imshow(water_mask, cmap="Blues")
axes[1, 0].set_title("water_mask (bit 7 de QA_PIXEL)")
axes[1, 0].axis("off")

axes[1, 1].imshow(rgb)
axes[1, 1].imshow(overlay_land)
axes[1, 1].set_title("land_mask sobre RGB\n(píxeles excluidos en amarillo)")
axes[1, 1].axis("off")

plt.tight_layout()
plt.show()



---

## 📘 Módulo 6 -- Control de validez numérica de la reflectancia

**Objetivo**  
Identificar píxeles con información óptica numéricamente válida.

Un píxel se considera válido cuando las bandas requeridas contienen valores finitos y distintos de `nodata`. Esta etapa no impone límites físicos estrictos de reflectancia.

**Salida**  
Máscara de validez numérica para los cálculos espectrales.

In [ ]:
# ============================================================
# MÓDULO 6 — Control de validez numérica de la reflectancia
# Objetivo:
# Construir la máscara de validez numérica de la reflectancia
# óptica corregida, para garantizar estabilidad en los cálculos
# espectrales posteriores.
#
# Resultado:
# - refl_valid_mask
# - land_refl_valid_mask
# ============================================================

# -----------------------------
# 1. Importación de librerías
# -----------------------------

# -----------------------------
# 2. Verificación de variables requeridas
# -----------------------------
required_vars = [
    "B2", "B3", "B4", "B5", "B6", "B7",
    "land_mask"
]

for var_name in required_vars:
    if var_name not in globals():
        raise NameError(f"No existe la variable requerida: {var_name}")

# -----------------------------
# 3. Verificación de compatibilidad dimensional
# -----------------------------
ref_shape = B2.shape

for band_name in ["B3", "B4", "B5", "B6", "B7"]:
    if globals()[band_name].shape != ref_shape:
        raise ValueError(f"La banda {band_name} no coincide espacialmente con B2.")

if land_mask.shape != ref_shape:
    raise ValueError("land_mask no coincide espacialmente con las bandas ópticas.")

# -----------------------------
# 4. Construcción de la máscara de validez radiométrica
#    (consistencia numérica en todas las bandas)
# -----------------------------
refl_valid_mask = (
    np.isfinite(B2) &
    np.isfinite(B3) &
    np.isfinite(B4) &
    np.isfinite(B5) &
    np.isfinite(B6) &
    np.isfinite(B7)
)

if "refl_nodata" in globals() and refl_nodata is not None:
    refl_valid_mask &= (
        (B2 != refl_nodata) &
        (B3 != refl_nodata) &
        (B4 != refl_nodata) &
        (B5 != refl_nodata) &
        (B6 != refl_nodata) &
        (B7 != refl_nodata)
    )

# -----------------------------
# 5. Máscara conjunta para uso espectral posterior
# -----------------------------
land_refl_valid_mask = land_mask & refl_valid_mask

# -----------------------------
# 6. Estadísticas de control
# -----------------------------
n_refl_valid = np.sum(refl_valid_mask)
n_refl_invalid = np.sum(~refl_valid_mask)
n_land = np.sum(land_mask)
n_land_refl_valid = np.sum(land_refl_valid_mask)

print("=" * 72)
print("MÓDULO 6 — VALIDEZ NUMÉRICA DE LA REFLECTANCIA")
print("=" * 72)

print(f"Shape de bandas ópticas            : {ref_shape}")
print(f"Píxeles válidos (refl_valid_mask)  : {n_refl_valid:,}")
print(f"Píxeles inválidos                  : {n_refl_invalid:,}")

print("\nResumen de máscaras:")
print(f"  land_mask = 1                    : {n_land:,}")
print(f"  refl_valid_mask = 1              : {n_refl_valid:,}")
print(f"  land_refl_valid_mask = 1         : {n_land_refl_valid:,}")

print("\nInterpretación:")
print("  refl_valid_mask      -> validez numérica de reflectancia")
print("  land_refl_valid_mask -> tierra válida + reflectancia válida")


# -----------------------------
# 8. Visualización de máscaras
# -----------------------------
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(land_mask, cmap="gray")
axes[0].set_title("land_mask")
axes[0].axis("off")

axes[1].imshow(refl_valid_mask, cmap="gray")
axes[1].set_title("refl_valid_mask")
axes[1].axis("off")

axes[2].imshow(land_refl_valid_mask, cmap="gray")
axes[2].set_title("land_refl_valid_mask")
axes[2].axis("off")

plt.tight_layout()
plt.show()

n_land_only_invalid_refl = np.sum(land_mask & (~refl_valid_mask))
print(f"Píxeles terrestres excluidos por invalidez numérica: {n_land_only_invalid_refl:,}")

---

## 📘 Módulo 7 -- Cálculo de NDVI

**Objetivo**  
Calcular el índice de vegetación de diferencia normalizada.

$$
NDVI=\frac{B5-B4}{B5+B4}
$$

El cálculo se restringe a píxeles válidos y con denominador numéricamente estable.

**Salida**  
Mapa de NDVI en el intervalo $[-1,1]$.

In [ ]:
# ============================================================
# MÓDULO 7 — Cálculo de NDVI
# Objetivo:
# Calcular el índice de vegetación de diferencia normalizada
# (NDVI) a partir de las bandas B5 (NIR) y B4 (Red),
# restringiendo el cálculo a píxeles terrestres con reflectancia
# numéricamente válida.
#
# Resultado:
# - ndvi
# - ndvi_valid_mask
# - ndvi_profile
# ============================================================

# -----------------------------
# 1. Importación de librerías
# -----------------------------

# -----------------------------
# 2. Verificación de variables requeridas
# -----------------------------
required_vars = ["B4", "B5", "land_refl_valid_mask", "refl_profile"]

for var_name in required_vars:
    if var_name not in globals():
        raise NameError(f"No existe la variable requerida: {var_name}")

if B4.shape != B5.shape:
    raise ValueError("B4 y B5 no tienen la misma dimensión.")

if land_refl_valid_mask.shape != B4.shape:
    raise ValueError("land_refl_valid_mask no coincide espacialmente con B4 y B5.")

# -----------------------------
# 3. Máscara válida para NDVI
#    Se exige:
#    - tierra válida
#    - reflectancia válida
#    - denominador distinto de cero
# -----------------------------
ndvi_valid_mask = (
    land_refl_valid_mask &
    np.isfinite(B4) &
    np.isfinite(B5) &
    (np.abs(B5 + B4) > 1e-12)
)

# -----------------------------
# 4. Cálculo de NDVI
# -----------------------------
ndvi = np.full(B4.shape, np.nan, dtype=np.float32)

ndvi[ndvi_valid_mask] = (
    (B5[ndvi_valid_mask] - B4[ndvi_valid_mask]) /
    (B5[ndvi_valid_mask] + B4[ndvi_valid_mask])
).astype(np.float32)

# Recorte de seguridad al rango físico esperado
ndvi = np.clip(ndvi, -1.0, 1.0)

# -----------------------------
# 5. Estadísticas básicas
# -----------------------------
valid_values = ndvi[ndvi_valid_mask]

print("=" * 72)
print("MÓDULO 7 — CÁLCULO DE NDVI")
print("=" * 72)

print(f"Shape NDVI                         : {ndvi.shape}")
print(f"Píxeles válidos NDVI               : {np.sum(ndvi_valid_mask):,}")
print(f"Píxeles inválidos NDVI             : {np.sum(~ndvi_valid_mask):,}")

if valid_values.size == 0:
    raise ValueError("No se obtuvieron píxeles válidos para el cálculo de NDVI.")

print("\nEstadísticas de NDVI:")
print(f"  Mínimo                           : {np.nanmin(valid_values):.6f}")
print(f"  Máximo                           : {np.nanmax(valid_values):.6f}")
print(f"  Media                            : {np.nanmean(valid_values):.6f}")
print(f"  Desv. estándar                   : {np.nanstd(valid_values):.6f}")
print(f"  Percentil 2                      : {np.nanpercentile(valid_values, 2):.6f}")
print(f"  Percentil 98                     : {np.nanpercentile(valid_values, 98):.6f}")

# -----------------------------
# 6. Perfil para eventual exportación
# -----------------------------
ndvi_profile = refl_profile.copy()
ndvi_profile.update(
    dtype="float32",
    count=1,
    nodata=np.nan
)

# -----------------------------
# 7. Visualización
# -----------------------------
plt.figure(figsize=(8, 7))
im = plt.imshow(ndvi, cmap="RdYlGn", vmin=-1, vmax=1)
plt.title("NDVI")
plt.axis("off")
plt.colorbar(im, fraction=0.046, pad=0.04, label="NDVI")
plt.show()



---

## 📘 Módulo 8 -- Fracción de vegetación ($P_v$)

**Objetivo**  
Estimar la fracción de vegetación a partir de la distribución de NDVI.

$$
P_v=
\left(
\frac{NDVI-NDVI_{\min}}
{NDVI_{\max}-NDVI_{\min}}
\right)^2
$$

Los valores $NDVI_{\min}$ y $NDVI_{\max}$ se obtienen mediante percentiles robustos y actúan como referencias aproximadas de los extremos suelo–vegetación presentes en la escena.

**Salida**  
Mapa de $P_v$ en el intervalo $[0,1]$.

In [ ]:
# ============================================================
# MÓDULO 8 — Fracción de vegetación (Pv)
# Objetivo:
# Estimar la fracción de vegetación a partir del NDVI usando
# límites robustos definidos por percentiles y un recorte previo
# del índice para reducir el efecto de valores extremos.
#
# Resultado:
# - ndvi_min
# - ndvi_max
# - ndvi_clip
# - Pv
# - pv_profile
# ============================================================

# -----------------------------
# 1. Importación de librerías
# -----------------------------

# -----------------------------
# 2. Verificación de variables requeridas
# -----------------------------
required_vars = ["ndvi", "ndvi_valid_mask", "ndvi_profile"]

for var_name in required_vars:
    if var_name not in globals():
        raise NameError(f"No existe la variable requerida: {var_name}")

if ndvi.shape != ndvi_valid_mask.shape:
    raise ValueError("ndvi y ndvi_valid_mask no tienen la misma dimensión.")

# -----------------------------
# 3. Extracción de valores válidos de NDVI
# -----------------------------
ndvi_values = ndvi[ndvi_valid_mask]

if ndvi_values.size == 0:
    raise ValueError("No existen valores válidos de NDVI para calcular Pv.")

# -----------------------------
# 4. Determinación de límites robustos
# -----------------------------
ndvi_min = np.nanpercentile(ndvi_values, 2)
ndvi_max = np.nanpercentile(ndvi_values, 98)

if not np.isfinite(ndvi_min) or not np.isfinite(ndvi_max):
    raise ValueError("ndvi_min o ndvi_max no son finitos.")

if ndvi_max <= ndvi_min:
    raise ValueError(
        f"Rango inválido para Pv: ndvi_max ({ndvi_max}) <= ndvi_min ({ndvi_min})."
    )

# -----------------------------
# 5. Recorte del NDVI
# -----------------------------
ndvi_clip = np.full(ndvi.shape, np.nan, dtype=np.float32)
ndvi_clip[ndvi_valid_mask] = np.clip(ndvi[ndvi_valid_mask], ndvi_min, ndvi_max).astype(np.float32)

# -----------------------------
# 6. Cálculo de fracción de vegetación
# -----------------------------
Pv = np.full(ndvi.shape, np.nan, dtype=np.float32)

Pv[ndvi_valid_mask] = (
    ((ndvi_clip[ndvi_valid_mask] - ndvi_min) / (ndvi_max - ndvi_min)) ** 2
).astype(np.float32)

# Recorte de seguridad al rango físico esperado
Pv = np.clip(Pv, 0.0, 1.0)

# -----------------------------
# 7. Estadísticas básicas
# -----------------------------
pv_values = Pv[ndvi_valid_mask]

print("=" * 72)
print("MÓDULO 8 — FRACCIÓN DE VEGETACIÓN (Pv)")
print("=" * 72)

print(f"Shape Pv                           : {Pv.shape}")
print(f"Píxeles válidos Pv                 : {np.sum(ndvi_valid_mask):,}")
print(f"Píxeles inválidos Pv               : {np.sum(~ndvi_valid_mask):,}")

print("\nLímites robustos del NDVI:")
print(f"  NDVI_min (P2)                    : {ndvi_min:.6f}")
print(f"  NDVI_max (P98)                   : {ndvi_max:.6f}")

print("\nEstadísticas de Pv:")
print(f"  Mínimo                           : {np.nanmin(pv_values):.6f}")
print(f"  Máximo                           : {np.nanmax(pv_values):.6f}")
print(f"  Media                            : {np.nanmean(pv_values):.6f}")
print(f"  Desv. estándar                   : {np.nanstd(pv_values):.6f}")

# -----------------------------
# 8. Perfil para eventual exportación
# -----------------------------
pv_profile = ndvi_profile.copy()
pv_profile.update(
    dtype="float32",
    count=1,
    nodata=np.nan
)

# -----------------------------
# 9. Visualización de Pv
# -----------------------------
plt.figure(figsize=(8, 7))
im = plt.imshow(Pv, cmap="YlGn", vmin=0, vmax=1)
plt.title("Fracción de vegetación (Pv)")
plt.axis("off")
plt.colorbar(im, fraction=0.046, pad=0.04, label="Pv")
plt.show()



---

## 📘 Módulo 9 -- Estimación de emisividad superficial

**Objetivo**  
Estimar la emisividad térmica efectiva a partir de la fracción de vegetación.

$$
\varepsilon=
\varepsilon_vP_v+\varepsilon_s(1-P_v)+C
$$

con:

$$
C=0.06P_v(1-P_v)
$$

Los valores $\varepsilon_v$ y $\varepsilon_s$ son parámetros del modelo para vegetación y suelo.

**Salida**  
Mapa de emisividad superficial.

In [ ]:
# ============================================================
# MÓDULO 9 — Estimación de emisividad superficial
# Objetivo:
# Estimar la emisividad superficial a partir de la fracción de
# vegetación (Pv) mediante un modelo de mezcla suelo-vegetación
# con corrección por efecto de cavidad en píxeles mixtos.
#
# Resultado:
# - emissivity
# - emissivity_profile
# ============================================================

# -----------------------------
# 1. Importación de librerías
# -----------------------------

# -----------------------------
# 2. Verificación de variables requeridas
# -----------------------------
required_vars = ["Pv", "ndvi_valid_mask", "pv_profile"]

for var_name in required_vars:
    if var_name not in globals():
        raise NameError(f"No existe la variable requerida: {var_name}")

if Pv.shape != ndvi_valid_mask.shape:
    raise ValueError("Pv y ndvi_valid_mask no tienen la misma dimensión.")

# -----------------------------
# 3. Parámetros del modelo
# -----------------------------
epsilon_v = 0.985   # emisividad de vegetación
epsilon_s = 0.960   # emisividad de suelo desnudo

# -----------------------------
# 4. Cálculo del término de cavidad
# -----------------------------
C = np.full(Pv.shape, np.nan, dtype=np.float32)
C[ndvi_valid_mask] = (0.06 * Pv[ndvi_valid_mask] * (1.0 - Pv[ndvi_valid_mask])).astype(np.float32)

# -----------------------------
# 5. Cálculo de emisividad superficial
# -----------------------------
emissivity = np.full(Pv.shape, np.nan, dtype=np.float32)

emissivity[ndvi_valid_mask] = (
    epsilon_v * Pv[ndvi_valid_mask] +
    epsilon_s * (1.0 - Pv[ndvi_valid_mask]) +
    C[ndvi_valid_mask]
).astype(np.float32)

# Recorte de seguridad a rango físico razonable
emissivity = np.clip(emissivity, 0.90, 1.00)

# -----------------------------
# 6. Estadísticas básicas
# -----------------------------
emis_values = emissivity[ndvi_valid_mask]

if emis_values.size == 0:
    raise ValueError("No existen valores válidos de emisividad.")

print("=" * 72)
print("MÓDULO 9 — ESTIMACIÓN DE EMISIVIDAD SUPERFICIAL")
print("=" * 72)

print(f"Shape emissivity                   : {emissivity.shape}")
print(f"Píxeles válidos                    : {np.sum(ndvi_valid_mask):,}")
print(f"Píxeles inválidos                  : {np.sum(~ndvi_valid_mask):,}")

print("\nParámetros del modelo:")
print(f"  epsilon_v                        : {epsilon_v:.3f}")
print(f"  epsilon_s                        : {epsilon_s:.3f}")

print("\nEstadísticas de emisividad:")
print(f"  Mínimo                           : {np.nanmin(emis_values):.6f}")
print(f"  Máximo                           : {np.nanmax(emis_values):.6f}")
print(f"  Media                            : {np.nanmean(emis_values):.6f}")
print(f"  Desv. estándar                   : {np.nanstd(emis_values):.6f}")

# -----------------------------
# 7. Perfil para eventual exportación
# -----------------------------
emissivity_profile = pv_profile.copy()
emissivity_profile.update(
    dtype="float32",
    count=1,
    nodata=np.nan
)

# -----------------------------
# 8. Visualización de emisividad
# -----------------------------
plt.figure(figsize=(8, 7))
im = plt.imshow(emissivity, vmin=0.96, vmax=0.99)
plt.title("Emisividad superficial")
plt.axis("off")
plt.colorbar(im, fraction=0.046, pad=0.04, label="Emisividad")
plt.show()



---

## 📘 Módulo 10 -- Parámetros atmosféricos

**Objetivo**  
Preparar los términos atmosféricos requeridos por la ecuación de transferencia radiativa.

**Entradas**
- `ST_URAD`: radiancia ascendente;
- `ST_DRAD`: radiancia descendente;
- `ST_ATRAN`: transmitancia atmosférica.

**Proceso**
Se aplican los factores de escala y se verifica la compatibilidad espacial de los tres productos.

**Salida**  
$L_{\uparrow}$, $L_{\downarrow}$ y $\tau$ en unidades físicas.

In [ ]:
# ============================================================
# MÓDULO 10 — Parámetros atmosféricos
# Objetivo:
# Recuperar los parámetros atmosféricos necesarios para la
# ecuación de transferencia radiativa a partir del producto
# Landsat Collection 2 Level-2 y convertirlos a unidades físicas.
#
# Resultado:
# - urad_dn, drad_dn, atran_dn
# - L_up
# - L_down
# - tau
# - atm_valid_mask
# - atm_profile
# ============================================================

# -----------------------------
# 1. Importación de librerías
# -----------------------------

# -----------------------------
# 2. Verificación de variables requeridas
# -----------------------------
required_vars = [
    "st_urad_path", "st_drad_path", "st_atran_path",
    "refl_profile", "land_mask"
]

for var_name in required_vars:
    if var_name not in globals():
        raise NameError(f"No existe la variable requerida: {var_name}")

# -----------------------------
# 3. Lectura y verificación espacial
# -----------------------------
with rasterio.open(st_urad_path) as src_urad, \
     rasterio.open(st_drad_path) as src_drad, \
     rasterio.open(st_atran_path) as src_atran:

    # Verificación entre bandas atmosféricas
    if src_urad.crs != src_drad.crs or src_urad.crs != src_atran.crs:
        raise ValueError("ST_URAD, ST_DRAD y ST_ATRAN no comparten el mismo CRS.")

    if src_urad.transform != src_drad.transform or src_urad.transform != src_atran.transform:
        raise ValueError("ST_URAD, ST_DRAD y ST_ATRAN no comparten la misma transform.")

    if (src_urad.height, src_urad.width) != (src_drad.height, src_drad.width) or \
       (src_urad.height, src_urad.width) != (src_atran.height, src_atran.width):
        raise ValueError("ST_URAD, ST_DRAD y ST_ATRAN no comparten el mismo shape.")

    # Verificación contra el grid de reflectancia/pipeline
    if src_urad.crs != refl_profile["crs"]:
        raise ValueError("Los parámetros atmosféricos no comparten CRS con el grid del pipeline.")

    if src_urad.transform != refl_profile["transform"]:
        raise ValueError("Los parámetros atmosféricos no comparten transform con el grid del pipeline.")

    if (src_urad.height != refl_profile["height"]) or (src_urad.width != refl_profile["width"]):
        raise ValueError("Los parámetros atmosféricos no comparten shape con el grid del pipeline.")

    # Lectura de DN
    urad_dn = src_urad.read(1).astype(np.float32)
    drad_dn = src_drad.read(1).astype(np.float32)
    atran_dn = src_atran.read(1).astype(np.float32)

    # Metadatos
    atm_profile = src_urad.profile.copy()
    urad_nodata = src_urad.nodata
    drad_nodata = src_drad.nodata
    atran_nodata = src_atran.nodata

# -----------------------------
# 4. Máscara de validez atmosférica
# -----------------------------
atm_valid_mask = (
    np.isfinite(urad_dn) &
    np.isfinite(drad_dn) &
    np.isfinite(atran_dn)
)

if urad_nodata is not None:
    atm_valid_mask &= (urad_dn != urad_nodata)

if drad_nodata is not None:
    atm_valid_mask &= (drad_dn != drad_nodata)

if atran_nodata is not None:
    atm_valid_mask &= (atran_dn != atran_nodata)

# -----------------------------
# 5. Conversión a unidades físicas
#    ST_URAD  -> factor 0.001
#    ST_DRAD  -> factor 0.001
#    ST_ATRAN -> factor 0.0001
# -----------------------------
L_up = np.full(urad_dn.shape, np.nan, dtype=np.float32)
L_down = np.full(drad_dn.shape, np.nan, dtype=np.float32)
tau = np.full(atran_dn.shape, np.nan, dtype=np.float32)

L_up[atm_valid_mask] = urad_dn[atm_valid_mask] * 0.001
L_down[atm_valid_mask] = drad_dn[atm_valid_mask] * 0.001
tau[atm_valid_mask] = atran_dn[atm_valid_mask] * 0.0001


# Control físico básico de los parámetros atmosféricos
atm_physical_mask = (
    atm_valid_mask &
    np.isfinite(L_up) &
    np.isfinite(L_down) &
    np.isfinite(tau) &
    (L_up >= 0) &
    (L_down >= 0) &
    (tau > 0) &
    (tau <= 1)
)

# -----------------------------
# 6. Máscara válida conjunta para uso posterior
#    (dominio terrestre + reflectancia válida + atmósfera válida)
# -----------------------------
if "ndvi_valid_mask" in globals():
    thermo_valid_mask = ndvi_valid_mask & atm_physical_mask
else:
    thermo_valid_mask = land_mask & atm_physical_mask

# -----------------------------
# 7. Estadísticas de control
# -----------------------------
L_up_vals = L_up[atm_valid_mask]
L_down_vals = L_down[atm_valid_mask]
tau_vals = tau[atm_valid_mask]

print("=" * 72)
print("MÓDULO 10 — PARÁMETROS ATMOSFÉRICOS")
print("=" * 72)

print(f"Shape parámetros atmosféricos      : {urad_dn.shape}")
print(f"Píxeles válidos atmósfera          : {np.sum(atm_valid_mask):,}")
print(f"Píxeles inválidos atmósfera        : {np.sum(~atm_valid_mask):,}")

print("\nNodata originales:")
print(f"  ST_URAD nodata                   : {urad_nodata}")
print(f"  ST_DRAD nodata                   : {drad_nodata}")
print(f"  ST_ATRAN nodata                  : {atran_nodata}")

print("\nRango en unidades físicas:")
print(f"  L_up   min / max                 : {np.nanmin(L_up_vals):.6f} / {np.nanmax(L_up_vals):.6f}")
print(f"  L_down min / max                 : {np.nanmin(L_down_vals):.6f} / {np.nanmax(L_down_vals):.6f}")
print(f"  tau    min / max                 : {np.nanmin(tau_vals):.6f} / {np.nanmax(tau_vals):.6f}")

print("\nMáscaras disponibles:")
print(f"  atm_valid_mask = 1               : {np.sum(atm_valid_mask):,}")
print(f"  atm_physical_mask = 1            : {np.sum(atm_physical_mask):,}")
print(f"  thermo_valid_mask = 1            : {np.sum(thermo_valid_mask):,}")

# -----------------------------
# 8. Perfiles para eventual exportación
# -----------------------------
atm_profile_float = atm_profile.copy()
atm_profile_float.update(
    dtype="float32",
    count=1,
    nodata=np.nan
)

# -----------------------------
# 9. Visualización rápida
# -----------------------------
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

im1 = axes[0].imshow(L_up)
axes[0].set_title("L_up")
axes[0].axis("off")
plt.colorbar(im1, ax=axes[0], fraction=0.046, pad=0.04)

im2 = axes[1].imshow(L_down)
axes[1].set_title("L_down")
axes[1].axis("off")
plt.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04)

im3 = axes[2].imshow(tau, vmin=0, vmax=1)
axes[2].set_title("tau")
axes[2].axis("off")
plt.colorbar(im3, ax=axes[2], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()



---

## 📘 Módulo 11 -- Radiancia térmica de superficie

**Objetivo**  
Recuperar la radiancia emitida por la superficie corrigiendo los efectos atmosféricos y la radiancia descendente reflejada.

$$
L_{\lambda}^{T}
=
\frac{
L_{\lambda}^{sen}
-
L_{\uparrow}
-
\tau(1-\varepsilon)L_{\downarrow}
}{
\tau\varepsilon
}
$$

El cálculo se limita a píxeles con parámetros atmosféricos y de emisividad finitos y físicamente válidos.

**Salida**  
Radiancia térmica atribuida a la superficie terrestre.

In [ ]:
# ============================================================
# MÓDULO 11 — Radiancia térmica de superficie
# Objetivo:
# Estimar la radiancia térmica emitida por la superficie a partir
# de la ecuación de transferencia radiativa (RTE).
#
# Ecuación:
# L_lambda_T = (L_lambda_sen - L_up - tau*(1-epsilon)*L_down) / (tau*epsilon)
#
# Resultado:
# - L_lambda_T
# - rte_denom
# - thermal_radiance_profile
# ============================================================

# -----------------------------
# 1. Importación de librerías
# -----------------------------

# -----------------------------
# 2. Verificación de variables requeridas
# -----------------------------
required_vars = [
    "L_lambda_sen",
    "emissivity",
    "L_up",
    "L_down",
    "tau",
    "thermo_valid_mask",
    "b10_profile"
]

for var_name in required_vars:
    if var_name not in globals():
        raise NameError(f"No existe la variable requerida: {var_name}")

# -----------------------------
# 3. Verificación de compatibilidad dimensional
# -----------------------------
ref_shape = L_lambda_sen.shape

for arr_name in ["emissivity", "L_up", "L_down", "tau", "thermo_valid_mask"]:
    if globals()[arr_name].shape != ref_shape:
        raise ValueError(f"{arr_name} no coincide espacialmente con L_lambda_sen.")

# -----------------------------
# 4. Denominador de la RTE
#    tau * epsilon
# -----------------------------
rte_denom = np.full(ref_shape, np.nan, dtype=np.float32)
rte_denom[thermo_valid_mask] = (
    tau[thermo_valid_mask] * emissivity[thermo_valid_mask]
).astype(np.float32)

# Máscara final estricta para RTE
rte_valid_mask = (
    thermo_valid_mask &
    np.isfinite(L_lambda_sen) &
    np.isfinite(emissivity) &
    np.isfinite(L_up) &
    np.isfinite(L_down) &
    np.isfinite(tau) &
    np.isfinite(rte_denom) &
    (tau > 0) &
    (tau <= 1) &
    (emissivity > 0) &
    (emissivity <= 1) &
    (rte_denom > 0)
)

# -----------------------------
# 5. Cálculo de radiancia térmica de superficie
# -----------------------------
L_lambda_T = np.full(ref_shape, np.nan, dtype=np.float32)

L_lambda_T[rte_valid_mask] = (
    (
        L_lambda_sen[rte_valid_mask]
        - L_up[rte_valid_mask]
        - tau[rte_valid_mask] * (1.0 - emissivity[rte_valid_mask]) * L_down[rte_valid_mask]
    )
    / rte_denom[rte_valid_mask]
).astype(np.float32)

# -----------------------------
# 6. Estadísticas básicas
# -----------------------------
Lsurf_vals = L_lambda_T[rte_valid_mask]

if Lsurf_vals.size == 0:
    raise ValueError("No existen píxeles válidos para calcular L_lambda_T.")

print("=" * 72)
print("MÓDULO 11 — RADIANCIA TÉRMICA DE SUPERFICIE")
print("=" * 72)

print(f"Shape L_lambda_T                   : {L_lambda_T.shape}")
print(f"Píxeles válidos RTE                : {np.sum(rte_valid_mask):,}")
print(f"Píxeles inválidos RTE              : {np.sum(~rte_valid_mask):,}")

print("\nEstadísticas de denominador tau*epsilon:")
print(f"  Mínimo                           : {np.nanmin(rte_denom[rte_valid_mask]):.6f}")
print(f"  Máximo                           : {np.nanmax(rte_denom[rte_valid_mask]):.6f}")

print("\nEstadísticas de L_lambda_T:")
print(f"  Mínimo                           : {np.nanmin(Lsurf_vals):.6f}")
print(f"  Máximo                           : {np.nanmax(Lsurf_vals):.6f}")
print(f"  Media                            : {np.nanmean(Lsurf_vals):.6f}")
print(f"  Desv. estándar                   : {np.nanstd(Lsurf_vals):.6f}")

# -----------------------------
# 7. Perfil para eventual exportación
# -----------------------------
thermal_radiance_profile = b10_profile.copy()
thermal_radiance_profile.update(
    dtype="float32",
    count=1,
    nodata=np.nan
)

# -----------------------------
# 8. Visualización
# -----------------------------
plt.figure(figsize=(8, 7))
im = plt.imshow(L_lambda_T)
plt.title("Radiancia térmica de superficie (L_lambda_T)")
plt.axis("off")
plt.colorbar(im, fraction=0.046, pad=0.04, label="W·m⁻²·sr⁻¹·µm⁻¹")
plt.show()



---

## 📘 Módulo 12 -- Conversión a temperatura superficial

**Objetivo**  
Convertir la radiancia térmica superficial en temperatura mediante la inversión de Planck.

$$
T=
\frac{K_2}
{\ln\left(\frac{K_1}{L_{\lambda}^{T}}+1\right)}
$$

y:

$$
LST_{^\circ C}=T-273.15
$$

**Salida**  
Mapa de LST en Kelvin y grados Celsius. La visualización utiliza percentiles robustos P2–P98 sin alterar los valores calculados.

In [ ]:
# ============================================================
# MÓDULO 12 — Conversión a temperatura superficial
# Objetivo:
# Convertir la radiancia térmica de superficie (L_lambda_T)
# en temperatura superficial mediante la forma invertida
# de la ecuación de Planck.
#
# Ecuación:
# LST_K = K2 / ln(K1 / L_lambda_T + 1)
#
# Resultado:
# - K1
# - K2
# - LST_K
# - LST_C
# - lst_profile
# ============================================================

# -----------------------------
# 1. Importación de librerías
# -----------------------------

# -----------------------------
# 2. Verificación de variables requeridas
# -----------------------------
required_vars = ["L_lambda_T", "thermal_radiance_profile"]

for var_name in required_vars:
    if var_name not in globals():
        raise NameError(f"No existe la variable requerida: {var_name}")

# -----------------------------
# 3. Recuperación de K1 y K2
#    desde el entorno (MTL)
# -----------------------------
candidate_k1_names = [
    "K1_CONSTANT_BAND_10",
    "k1_constant_band_10",
    "K1_B10",
    "k1_b10",
    "K1"
]

candidate_k2_names = [
    "K2_CONSTANT_BAND_10",
    "k2_constant_band_10",
    "K2_B10",
    "k2_b10",
    "K2"
]

K1 = None
K2 = None

for name in candidate_k1_names:
    if name in globals():
        K1 = float(globals()[name])
        break

for name in candidate_k2_names:
    if name in globals():
        K2 = float(globals()[name])
        break

if K1 is None or K2 is None:
    raise NameError(
        "No se encontraron las constantes térmicas K1 y K2 en el entorno."
    )

# -----------------------------
# 4. Conversión a temperatura
# -----------------------------
lst_valid_mask = np.isfinite(L_lambda_T) & (L_lambda_T > 0)
if "rte_valid_mask" in globals():
    lst_valid_mask &= rte_valid_mask

if not np.any(lst_valid_mask):
    raise ValueError("No existen píxeles físicamente válidos para invertir la función de Planck.")

LST_K = np.full(L_lambda_T.shape, np.nan, dtype=np.float32)
LST_C = np.full(L_lambda_T.shape, np.nan, dtype=np.float32)

LST_K[lst_valid_mask] = (
    K2 / np.log((K1 / L_lambda_T[lst_valid_mask]) + 1.0)
).astype(np.float32)

LST_C[lst_valid_mask] = (LST_K[lst_valid_mask] - 273.15).astype(np.float32)

# -----------------------------
# 5. Estadísticas básicas
# -----------------------------
print("=" * 72)
print("MÓDULO 12 — CONVERSIÓN A TEMPERATURA SUPERFICIAL")
print("=" * 72)

print(f"K1 utilizado                        : {K1:.6f}")
print(f"K2 utilizado                        : {K2:.6f}")

print(f"\nShape LST                           : {LST_K.shape}")

print("\nEstadísticas de LST en Kelvin:")
print(f"  Mínimo                            : {np.nanmin(LST_K):.6f}")
print(f"  Máximo                            : {np.nanmax(LST_K):.6f}")
print(f"  Media                             : {np.nanmean(LST_K):.6f}")
print(f"  Desv. estándar                    : {np.nanstd(LST_K):.6f}")

print("\nEstadísticas de LST en °C:")
print(f"  Mínimo                            : {np.nanmin(LST_C):.6f}")
print(f"  Máximo                            : {np.nanmax(LST_C):.6f}")
print(f"  Media                             : {np.nanmean(LST_C):.6f}")
print(f"  Desv. estándar                    : {np.nanstd(LST_C):.6f}")

# -----------------------------
# 6. Perfil para exportación
# -----------------------------
lst_profile = thermal_radiance_profile.copy()
lst_profile.update(
    dtype="float32",
    count=1,
    nodata=np.nan
)

# -----------------------------
# 7. Visualización
# -----------------------------
lst_values = LST_C[lst_valid_mask]
vmin = np.nanpercentile(lst_values, 2)
vmax = np.nanpercentile(lst_values, 98)

plt.figure(figsize=(8, 7))
im = plt.imshow(LST_C, cmap="turbo", vmin=vmin, vmax=vmax)
plt.title("Temperatura superficial (LST, °C)")
plt.axis("off")

cbar = plt.colorbar(im, fraction=0.046, pad=0.04)
cbar.set_label("°C")
plt.show()



## 📘 Módulo 13 -- Comparación de consistencia con Landsat Collection 2 Level-2

**Objetivo**  
Comparar la LST reconstruida con el producto oficial `ST_B10`.

**Alcance**  
La comparación evalúa coherencia numérica y espacial, pero no constituye una validación independiente, porque el pipeline utiliza parámetros atmosféricos derivados del mismo producto Level-2.

**Métricas**
- sesgo medio;
- MAE;
- RMSE;
- coeficiente de correlación;
- $R^2$;
- pendiente de la regresión;
- diferencia espacial:

$$
\Delta LST=LST_{\mathrm{pipeline}}-LST_{\mathrm{L2}}
$$

**Salida**  
Indicadores cuantitativos y mapas de diferencia para evaluar la consistencia del procedimiento.

In [ ]:
# ============================================================
# MÓDULO 13 — Comparación de consistencia con el producto oficial Level-2
# Objetivo:
# Comparar, como control de consistencia, la temperatura superficial
# estimada en el pipeline con ST_B10 de Landsat Collection 2 Level-2.
#
# Resultado:
# - ST_B10_K
# - ST_B10_C
# - compare_valid_mask
# - delta_LST
# ============================================================

# -----------------------------
# 1. Importación de librerías
# -----------------------------

# -----------------------------
# 2. Verificación de variables requeridas
# -----------------------------
required_vars = ["st_b10_path", "LST_C", "lst_valid_mask"]

for var_name in required_vars:
    if var_name not in globals():
        raise NameError(f"No existe la variable requerida: {var_name}")

if LST_C.shape != lst_valid_mask.shape:
    raise ValueError("LST_C y lst_valid_mask no tienen la misma dimensión.")

# -----------------------------
# 3. Lectura de ST_B10
# -----------------------------
with rasterio.open(st_b10_path) as src:
    st_b10_dn = src.read(1).astype(np.float32)
    st_b10_profile = src.profile.copy()
    st_b10_nodata = src.nodata

if st_b10_dn.shape != LST_C.shape:
    raise ValueError("ST_B10 no coincide espacialmente con LST_C.")

# -----------------------------
# 4. Máscara válida de ST_B10
# -----------------------------
st_valid_mask = np.isfinite(st_b10_dn)

if st_b10_nodata is not None:
    st_valid_mask &= (st_b10_dn != st_b10_nodata)

# -----------------------------
# 5. Conversión de ST_B10 a unidades físicas
#    Escala oficial:
#    ST_K = DN * 0.00341802 + 149.0
# -----------------------------
ST_B10_K = np.full(st_b10_dn.shape, np.nan, dtype=np.float32)
ST_B10_C = np.full(st_b10_dn.shape, np.nan, dtype=np.float32)

ST_B10_K[st_valid_mask] = st_b10_dn[st_valid_mask] * 0.00341802 + 149.0
ST_B10_C[st_valid_mask] = ST_B10_K[st_valid_mask] - 273.15

# -----------------------------
# 6. Dominio común de comparación
# -----------------------------
compare_valid_mask = (
    lst_valid_mask &
    st_valid_mask &
    np.isfinite(LST_C) &
    np.isfinite(ST_B10_C)
)

n_valid = np.sum(compare_valid_mask)
if n_valid == 0:
    raise ValueError("No existen píxeles válidos para comparar LST_C con ST_B10_C.")

# -----------------------------
# 7. Diferencia pixel a pixel
# -----------------------------
delta_LST = np.full(LST_C.shape, np.nan, dtype=np.float32)
delta_LST[compare_valid_mask] = (
    LST_C[compare_valid_mask] - ST_B10_C[compare_valid_mask]
).astype(np.float32)

# -----------------------------
# 8. Métricas de comparación
# -----------------------------
x = ST_B10_C[compare_valid_mask].astype(np.float64)   # referencia L2
y = LST_C[compare_valid_mask].astype(np.float64)      # pipeline
d = delta_LST[compare_valid_mask].astype(np.float64)

bias = np.mean(d)
mae = np.mean(np.abs(d))
rmse = np.sqrt(np.mean(d**2))
corr = np.corrcoef(x, y)[0, 1]

# Regresión lineal simple
slope, intercept = np.polyfit(x, y, 1)
r2 = corr**2

# -----------------------------
# 9. Estadísticas
# -----------------------------
print("=" * 72)
print("MÓDULO 13 — COMPARACIÓN DE CONSISTENCIA CON EL PRODUCTO OFICIAL L2")
print("=" * 72)

print(f"Píxeles válidos comparación        : {n_valid:,}")
print(f"ST_B10 nodata                      : {st_b10_nodata}")

print("\nRango de temperaturas (°C):")
print(f"  LST pipeline min / max           : {np.nanmin(y):.6f} / {np.nanmax(y):.6f}")
print(f"  ST_B10 L2  min / max             : {np.nanmin(x):.6f} / {np.nanmax(x):.6f}")

print("\nMétricas de comparación:")
print(f"  Bias (Pipeline - L2)             : {bias:.6f} °C")
print(f"  MAE                              : {mae:.6f} °C")
print(f"  RMSE                             : {rmse:.6f} °C")
print(f"  Correlación (r)                  : {corr:.6f}")
print(f"  R²                               : {r2:.6f}")
print(f"  Pendiente                        : {slope:.6f}")
print(f"  Intercepto                       : {intercept:.6f} °C")

print("\nEstadísticas de ΔLST:")
print(f"  Mínimo                           : {np.nanmin(d):.6f} °C")
print(f"  Máximo                           : {np.nanmax(d):.6f} °C")
print(f"  Media                            : {np.nanmean(d):.6f} °C")
print(f"  Desv. estándar                   : {np.nanstd(d):.6f} °C")

# -----------------------------
# 10. Scatter plot
# -----------------------------
# Muestreo opcional para no saturar la figura
max_points = 50000
if n_valid > max_points:
    rng = np.random.default_rng(42)
    idx = rng.choice(n_valid, size=max_points, replace=False)
    x_plot = x[idx]
    y_plot = y[idx]
else:
    x_plot = x
    y_plot = y

line_min = min(np.nanmin(x_plot), np.nanmin(y_plot))
line_max = max(np.nanmax(x_plot), np.nanmax(y_plot))

plt.figure(figsize=(7, 7))
plt.scatter(x_plot, y_plot, s=3, alpha=0.25)
plt.plot([line_min, line_max], [line_min, line_max], "--", linewidth=1.2, label="1:1")
plt.plot(
    [line_min, line_max],
    [slope * line_min + intercept, slope * line_max + intercept],
    linewidth=1.5,
    label="Regresión"
)
plt.xlabel("ST_B10 Level-2 (°C)")
plt.ylabel("LST pipeline (°C)")
plt.title("Consistencia entre LST del pipeline y ST_B10")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# -----------------------------
# 11. Mapa de diferencias
# -----------------------------
plt.figure(figsize=(8, 7))
im = plt.imshow(delta_LST, cmap="coolwarm", vmin=-5, vmax=5)
plt.title("ΔLST = LST_pipeline - ST_B10 (°C)")
plt.axis("off")
cbar = plt.colorbar(im, fraction=0.046, pad=0.04)
cbar.set_label("°C")
cbar.set_ticks([-5, -3, -1, 0, 1, 3, 5])
plt.show()

# -----------------------------
# 12. Histograma de diferencias
# -----------------------------
plt.figure(figsize=(8, 5))
plt.hist(d, bins=60)
plt.axvline(0, linestyle="--", linewidth=1)
plt.xlabel("ΔLST (°C)")
plt.ylabel("Frecuencia")
plt.title("Histograma de diferencias: LST_pipeline - ST_B10")
plt.grid(True, alpha=0.3)
plt.show()

